# Step 5: 端到端闭环 mini 项目 + 前瞻（finale）

**目标**：把整条 M2→M3→M4 链路串成端到端闭环——理解整条链路的**唯一交接物**（compressed-tensors 产物目录）、能按场景约束（显存/算力/精度/硬件）决策选方法（M1.6 三范式 + M3.8 四维 Pareto）、组装可复现**交付物四件套**（recipe + uv.lock + serve 命令 + 压测报告），并前瞻 QAT/NVFP4。

**对应 OUTLINE 课时**：4.6 端到端闭环 mini 项目（~90min）+ 4.7 QAT/NVFP4 前瞻（~25min，finale）。

> **闭环认知**：整条链路之所以能解耦（量化/调优/部署各模块独立），全靠那个**唯一的交接物**——compressed-tensors 产物目录（`config.json` 的 `quantization_config` + 量化权重）。M2 产出它、M3 调优它、M4 读它部署，全程不传 Python 对象、只传一个目录。


## 学完应能讲清（学完本节应能口头回答）

1. 整条链路的**唯一交接物**是什么？（compressed-tensors 产物目录：`config.json` 的 `quantization_config` + 量化权重）为什么这个交接物让量化/调优/部署解耦？（提示：各模块只读写一个目录，不传 Python 对象/不共享内存状态；M2 产、M3 调、M4 读）
2. 给场景约束（显存紧张 / 算力受限 / 精度优先 / H200），怎么结合 **M1.6 三范式**（W4A16 访存墙 / W8A8 计算墙 / FP8）+ **M3.8 四维 Pareto**（质量/显存/吞吐/TTFT）选方法？（提示：显存紧→W4A16、吞吐/精度优先→FP8、非 Hopper→W8A8）
3. 可复现**交付物四件套**是哪四件（recipe + 各模块 uv.lock + serve 命令 + 压测报告）？为什么固定 seed/prompt/batch？（提示：四件套锁定量化逻辑+环境版本+部署参数+性能数字；不固定则复现结果漂移，对比无意义）
4. QAT 为什么本课不动手（万亿 token 重训成本，仅蒸馏/继续训练）？NVFP4 为什么 H200 跑不了（Blackwell 第五代 Tensor Core 独有）？


In [ ]:
%%capture
import pathlib, os, json
import ipytest
ipytest.autoconfig()


In [ ]:
# Setup cell（cwd 无关路径解析）。M4 跨模块读 M2/M3 7B 量化产物做闭环串接。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT   = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT       = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_COURSE = MODULE_ROOT.parent
M2_OUT = REPO_COURSE / "m2-quant-pipeline" / "out"      # 唯一交接物：三方法产物目录
M3_OUT = REPO_COURSE / "m3-tuning-eval" / "out"
print("MODULE_ROOT =", MODULE_ROOT)
print("M2_OUT =", M2_OUT, "| exists:", M2_OUT.exists())


## 原理：唯一交接物 + 选型决策 + 交付物四件套

**整条链路的唯一交接物**：compressed-tensors **产物目录**（`config.json` 的 `quantization_config` + 量化权重 `.safetensors`）。这是 M2→M3→M4 解耦的关键——各模块只读写这一个目录，不传 Python 对象、不共享内存状态：

```
M2 量化     ──产出──▶  out/qwen7b-{fp8,awq,smoothquant}/   (config + 权重)
                              │ 唯一交接物（目录）
M3 调优     ──读改──▶  out/(layer fallback / mixed-precision)
                              │ 同一目录格式
M4 部署     ──读───▶  vllm serve <dir>   (vLLM 读 config.json 自动适配)
```

为什么这让链路解耦：M2 用 quant env（llmcompressor）、M4 用 vllm env，两个 env 的 transformers 版本都不同——但它们通过一个**磁盘上的目录**交接，互不干扰。换量化方法（M2）不动部署（M4）、调优（M3）不重量化（M2）。

**选型决策（M1.6 三范式 + M3.8 四维 Pareto 综合）**：

| 场景约束 | 推荐方法 | 理由（范式/维度）|
|---|---|---|
| 显存紧张 | AWQ (W4A16) | weight-only 4-bit 最省显存（访存墙方案）|
| 算力受限 | AWQ (W4A16) | 不依赖 Tensor Core 计算，算力弱卡友好 |
| 吞吐优先 / TTFT 敏感 | FP8 | Hopper FP8 Tensor Core 快 + 省显存让出 KV |
| 精度优先 | FP8 | 浮点抗离群点，精度≈FP16 |
| H200 生产默认 | FP8 | OUTLINE 1.6：FP8 全面替代 INT8-W8A8 |
| 非 Hopper（Ampere/Ada）| SmoothQuant (W8A8) | 无 FP8 Tensor Core，INT8 计算墙方案首选 |

**交付物四件套（可复现）**：
1. **recipe**：量化 recipe（如 `QuantizationModifier(targets='Linear', scheme='FP8_DYNAMIC', ignore=['lm_head'])`）——锁定量化逻辑。
2. **uv.lock**（各模块）：锁定环境版本（vllm/llmcompressor/compressed-tensors/transformers）——复现的关键（跨平台精确版本图）。
3. **serve 命令**：部署参数（`vllm serve <dir> --tensor-parallel-size 2 ...`）——锁定部署配置。
4. **压测报告**：性能数字（吞吐/TTFT/显存对比）——验证交付有效。

**为什么固定 seed/prompt/batch**：量化/调优/评测/压测每个环节都有随机性（校准采样、生成采样、压测请求分布）。不固定则每次复现结果漂移，"量化前后对比"无意义。固定 seed(42) + prompt 集 + batch/num_prompts 才能拿到可重复的对比。


## 亲手摸一摸：整条链路产物串接（M2/M3/M4）

看 M2 `out/` 的产物目录如何串到 M4——config.json + 量化权重就是那个唯一交接物。


In [ ]:
# 摸一摸：M2 out/ 的产物目录结构（唯一交接物）
products = {"FP8": M2_OUT / "qwen7b-fp8", "AWQ": M2_OUT / "qwen7b-awq", "SmoothQuant": M2_OUT / "qwen7b-smoothquant"}
print("=== 整条链路的唯一交接物：compressed-tensors 产物目录 ===")
for name, d in products.items():
    if not d.exists():
        print("[%s] %s 不存在（先跑 M2）" % (name, d))
        continue
    files = sorted(p.name for p in d.iterdir() if not p.name.startswith("."))
    has_config = (d / "config.json").exists()
    has_weights = any(f.endswith(".safetensors") for f in files)
    print("\n[%s] %s" % (name, d))
    print("  文件:", files[:6])
    print("  config.json 含 quantization_config:", has_config, "| 量化权重 .safetensors:", has_weights)
    print("  -> M4 只需 vllm serve %s 即可部署（读 config.json 自动适配）" % d)
print("\n=== 链路串接 ===")
print("  M2 量化   -> out/qwen7b-fp8/  (产 config + 权重)")
print("  M3 调优   -> 读改同一格式      (layer fallback / mixed-precision)")
print("  M4 部署   -> vllm serve <dir> (读 config.json auto 适配)")
print("  全程只交接「一个目录」，三模块解耦。")


## 本步填空（2 个）

1. **`build_decision_tree(constraints)`**（判断型）— 场景约束（显存/吞吐/精度/硬件）→ 推荐方法（FP8/AWQ/SmoothQuant）+ 理由。**为什么这么设计（填前先想）**：决策/判断型——综合 M1.6 三范式 + M3.8 四维 Pareto，是 finale 的核心交付（"给我场景我选路径"）。
2. **`build_delivery_bundle(recipe, uv_locks, serve_cmd, bench_report)`**（组装型）— 组装可复现交付物四件套 manifest（dict）。**为什么这么设计**：组装型——把零散交付物结构化，体现"可复现"的工程约定；缺件报错。


In [ ]:
def build_decision_tree(constraints):
    """场景约束 -> 推荐方法 + 理由（dict）。

    参数 constraints：约束标签的集合，可以是
      - list/tuple：['显存紧张', 'H200']
      - dict：{'显存紧张': True, 'H200': True} 或 {'显存紧张': 'KV 紧张'}

    为什么这么设计（填前先想）：
    - 决策型——综合 M1.6 三范式（W4A16 访存墙 / W8A8 计算墙 / FP8）+ M3.8 四维 Pareto。
    - 优先级链（先精确硬件，再算力/显存，再精度/吞吐；理由见 c04 表）：
        非Hopper > 显存紧张 > 算力受限 > 吞吐优先 > TTFT敏感 > 精度优先 > H200 > 默认(FP8)
    - 匹配到的最高优先约束决定推荐；返回 dict(recommended=, reason=, matched_constraint=)。
    - 默认（无任何已知约束）-> FP8（H200 生产默认）。

    返回：{'recommended': <方法>, 'reason': <理由>, 'matched_constraint': <命中的标签>}。
    """
    # TODO:
    #   1) 把 constraints 归一成标签 list（list 直接用；dict 取键）。
    #   2) 按优先级链遍历，第一个在标签里的就是命中。
    #   3) 查 DECISION_RULES（建议你定义在本函数外的模块级 dict：标签->(方法,理由)）取 (method, reason)。
    #   4) 返回 dict。无命中走 '默认推荐' -> FP8。
    #   提示：标签与方法映射见 c04 选型表，自己组装进 DECISION_RULES。
    raise NotImplementedError


In [ ]:
def build_delivery_bundle(recipe, uv_locks, serve_cmd, bench_report):
    """组装可复现交付物四件套 manifest（dict）。

    四件套参数：
    - recipe：量化 recipe（dict 或路径 str）—— 锁定量化逻辑
    - uv_locks：各模块 uv.lock（dict module->path）—— 锁定环境版本
    - serve_cmd：部署命令（str）—— 锁定部署配置
    - bench_report：压测报告（dict 或路径 str）—— 验证交付有效

    为什么这么设计（填前先想）：
    - 组装型——把零散交付物结构化成一份 manifest，体现"可复现"的工程约定。
    - 缺任一件（None 或空）-> ValueError，明确点出缺哪个（可复现不能有缺口）。
    - 加 bundle_version + repro_note（固定 seed/prompt/batch 说明）。

    返回：dict（含 recipe/uv_locks/serve_cmd/bench_report + bundle_version + repro_note）。
    """
    # TODO:
    #   1) 检查四件套是否齐全：None 或空（str/list/dict/tuple len 0）-> ValueError('交付物缺失：...')。
    #   2) 组 dict 返回（四件套 + 'bundle_version': '1.0' + repro_note）。
    raise NotImplementedError


In [ ]:
%%ipytest -qq

def test_decision_memory_tight():
    r = build_decision_tree(["显存紧张", "H200"])
    assert r["recommended"] == "AWQ (W4A16)", r
    assert "访存" in r["reason"] or "省显存" in r["reason"]
    assert r["matched_constraint"] == "显存紧张"

def test_decision_throughput():
    r = build_decision_tree(["吞吐优先"])
    assert r["recommended"] == "FP8"

def test_decision_precision():
    r = build_decision_tree(["精度优先"])
    assert r["recommended"] == "FP8"

def test_decision_non_hopper_overrides():
    # 非 Hopper 最优先，即使别的约束也在
    r = build_decision_tree(["非Hopper", "精度优先"])
    assert r["recommended"] == "SmoothQuant (W8A8)", r
    assert r["matched_constraint"] == "非Hopper"

def test_decision_dict_constraints():
    r = build_decision_tree({"显存紧张": True, "算力受限": "弱卡"})
    assert r["recommended"] == "AWQ (W4A16)"

def test_decision_default():
    r = build_decision_tree([])
    assert r["recommended"] == "FP8"
    assert r["matched_constraint"] == "默认推荐"

def test_delivery_bundle_complete():
    bundle = build_delivery_bundle(
        recipe={"scheme": "FP8_DYNAMIC", "targets": "Linear"},
        uv_locks={"m2": "course/m2-quant-pipeline/uv.lock", "m4": "course/m4-deploy-loop/uv.lock"},
        serve_cmd="vllm serve /models/qwen7b-fp8 --tensor-parallel-size 2",
        bench_report={"throughput": 1200.0, "ttft_ms": 45.0},
    )
    assert bundle["recipe"]["scheme"] == "FP8_DYNAMIC"
    assert "m4" in bundle["uv_locks"]
    assert "vllm serve" in bundle["serve_cmd"]
    assert bundle["bench_report"]["throughput"] == 1200.0
    assert bundle["bundle_version"] == "1.0"
    assert "seed" in bundle["repro_note"]

def test_delivery_bundle_missing_recipe():
    import pytest
    with pytest.raises(ValueError, match="recipe"):
        build_delivery_bundle(recipe=None,
            uv_locks={"m4": "x"}, serve_cmd="vllm serve x", bench_report={"t": 1})

def test_delivery_bundle_missing_serve_cmd():
    import pytest
    with pytest.raises(ValueError, match="serve_cmd"):
        build_delivery_bundle(recipe={"s": 1}, uv_locks={"m4": "x"}, serve_cmd="", bench_report={"t": 1})


## L2（CPU）：决策树 + 交付物组装（纯逻辑）

L2 验 `build_decision_tree` 选型 + `build_delivery_bundle` 组装（CPU 可跑，纯逻辑）。


In [ ]:
## L2：跑决策树 + 组装交付物四件套
print("=== L2：场景 -> 推荐方法 ===")
cases = [
    (["显存紧张", "H200"],   "显存紧 + H200"),
    (["吞吐优先"],           "吞吐优先"),
    (["非Hopper", "精度优先"], "非 Hopper（即使精度优先）"),
    ([],                     "无约束（默认）"),
]
for cons, label in cases:
    r = build_decision_tree(cons)
    print("  [%s] -> %s（理由：%s）" % (label, r["recommended"], r["reason"][:40]))

print("\n=== L2：组装交付物四件套 ===")
bundle = build_delivery_bundle(
    recipe={"scheme": "FP8_DYNAMIC", "targets": "Linear", "ignore": ["lm_head"]},
    uv_locks={"m2": "course/m2-quant-pipeline/uv.lock",
              "m3": "course/m3-tuning-eval/uv.lock",
              "m4": "course/m4-deploy-loop/uv.lock"},
    serve_cmd="vllm serve /models/qwen7b-fp8 --tensor-parallel-size 2 --enable-prefix-caching",
    bench_report={"throughput_tps": 1200.0, "ttft_ms": 45.0, "kv_cache_usage_perc": 0.42},
)
print("  bundle_version:", bundle["bundle_version"])
print("  recipe:", bundle["recipe"])
print("  uv_locks 模块:", list(bundle["uv_locks"].keys()))
print("  repro_note:", bundle["repro_note"])
json.dump(bundle, open(OUT_ROOT / "s5_delivery_bundle.json", "w"), indent=2, ensure_ascii=False)
assert build_decision_tree(["非Hopper"])["recommended"] == "SmoothQuant (W8A8)"
assert bundle["bundle_version"] == "1.0" and "seed" in bundle["repro_note"]
print("\nL2 通过：选型决策 + 四件套组装正确（产物存 out/s5_delivery_bundle.json）。")


## L3（可选）：链路串接演示

L3 可选：演示从 M2 产物 → M4 serve → 决策/组装的完整链路（纯逻辑已在 L1+L2 验，L3 留作真人串接演示）。

> **L3 双守卫**：`torch.cuda.is_available() and not os.environ.get('SKIP_L3')`——reviewer 执行验证设 `SKIP_L3=1` 跳过；真人跑时不设，L3 实证。


In [ ]:
import torch, os

def run_l3_e2e_link():
    # 演示：M2 产物 -> 决策 -> serve 命令 -> bundle（不真起服务，只串逻辑）
    products = {"FP8": M2_OUT / "qwen7b-fp8", "AWQ": M2_OUT / "qwen7b-awq",
                "SmoothQuant": M2_OUT / "qwen7b-smoothquant"}
    available = {n: p for n, p in products.items() if (p / "config.json").exists()}
    if not available:
        print("[L3] 无 M2 产物可串接（先跑 M2）。链路逻辑见 L2。")
        return
    print("[L3] 端到端链路串接（M2 产物 -> 决策 -> serve -> bundle）:")
    # 按场景挑一个方法部署
    decision = build_decision_tree(["H200", "吞吐优先"])
    method = decision["recommended"]
    print("  场景[H200,吞吐优先] -> 推荐", method)
    # 找对应产物（FP8->qwen7b-fp8 等）
    pick = {"FP8": "FP8", "AWQ (W4A16)": "AWQ", "SmoothQuant (W8A8)": "SmoothQuant"}
    prod = available.get(pick.get(method, "FP8"), list(available.values())[0])
    serve_cmd = "vllm serve %s --tensor-parallel-size 2 --enable-prefix-caching" % prod
    print("  serve:", serve_cmd)
    bundle = build_delivery_bundle(
        recipe={"scheme": method.split(" ")[0], "source": str(prod)},
        uv_locks={"m4": "course/m4-deploy-loop/uv.lock"},
        serve_cmd=serve_cmd,
        bench_report={"note": "见 s3 L3 四向压测"},
    )
    print("  bundle:", bundle["bundle_version"], "| repro:", bundle["repro_note"][:50])

if torch.cuda.is_available() and not os.environ.get('SKIP_L3'):
    run_l3_e2e_link()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（链路逻辑 L1+L2 已验；真人跑时不设 SKIP_L3 可串接演示）。")


## 前瞻：QAT 与 NVFP4（纯概念，不填空）

整条 M2→M3→M4 闭环到此走通。最后前瞻两个"本课不动手但要知道方向"的技术：

### QAT（Quantization-Aware Training）
- **思想**：训练时插入 fake-quant（前向模拟量化截断）+ **直通估计器（STE，Straight-Through Estimator）**（反向把量化不可导的梯度"直通"过去），让模型在训练中就适应量化误差——理论精度上限高于 PTQ。
- **为什么本课不动手**：LLM 的 QAT 需要在**接近原始训练规模的数据（万亿 token）**上重新训练（或继续训练），成本对工业级 7B+ 模型不可接受。QAT 仅在**蒸馏 / 继续训练**场景（已有训练 pipeline、只做量化适配微调）才划算。本课定位 PTQ 工业落地，故 QAT 仅作概念。
- **方向**：未来若开源大模型 QAT 后的检查点普及（如社区产 QAT 版 Qwen/Llama），可直接拿来部署——但那是消费别人的 QAT 产物，不是自己跑 QAT。

### NVFP4 / FP4
- **是什么**：4-bit 浮点（NVFP4 微缩浮点格式），**Blackwell（B100/B200）第五代 Tensor Core 独有**。
- **为什么 H200 跑不了**：Hopper（H200/H100）只有 FP8 Tensor Core，**没有 FP4 Tensor Core**——零 FP4 吞吐。NVFP4 是 Blackwell 世代（2025+）的硬件红利，H200 上跑 FP4 只能软件模拟（慢）。
- **方向**：当 Blackwell 数据中心卡普及，FP4（比 FP8 再省一半显存 + 更高吞吐）会成为新首选——本课的 FP8 流水线可平滑迁移（同样的 compressed-tensors 产物格式 + vLLM 声明式部署，只是 scheme 换 FP4、kernel 换 Blackwell FP4）。

> **产物可发布到 HF Hub**（OUTLINE 4.5 降级提示）：本课产出的 compressed-tensors 产物目录可用 `huggingface_hub` 的 `upload_folder` 发布到 Hub 供他人复现/部署。本课不实操（Hub 发布偏运维/分享动作、非核心部署能力）。

---

**课程闭环达成**：从 M1 激活离群点原理 → M2 三方法量化流水线 → M3 精度调优 + 评测 → M4 vLLM 声明式部署 + 闭环，你已具备在 H200×8 上把一个 7B 模型量化、调优、部署、压测、排错的完整工业能力。**唯一交接物**（compressed-tensors 产物目录）+ **交付物四件套**（recipe + uv.lock + serve + bench）= 可复现的端到端量化部署流水线。
